# BioGym Arena: source-backed LLM RL pilot
Runs a Qwen3-0.6B LoRA GRPO pilot on the cached UniProt/RCSB suite. This is **not Sarvam 30B training**. Use a GPU runtime; the current account enables T4. Review `docs/rl-training.md` for the split, reward, and provenance rules.


In [ ]:
import torch, subprocess
print(subprocess.check_output(['nvidia-smi'], text=True))
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime before training'


In [ ]:
!git clone --depth 1 --branch feat/application-v1 https://github.com/Ajey95/biogym-arena.git /content/biogym-arena
%cd /content/biogym-arena
%pip install -q -e 'backend[train]'


In [ ]:
from biogym.rl_suite import SUITE_HASH, split_tasks
print('Suite:', SUITE_HASH)
print({name: len(split_tasks(name)) for name in ('train', 'test', 'transfer')})
assert len(split_tasks('train')) == 64 and len(split_tasks('test')) == 16
!python -m biogym.rl_train --output /content/biogym-run --model Qwen/Qwen3-0.6B --max-steps 16


In [ ]:
import json
from pathlib import Path
report = json.loads(Path('/content/biogym-run/run_manifest.json').read_text())
print('Model:', report['model_id'], 'optimizer steps:', report['optimizer_steps'])
print('Before:', report['before_test']['successes'], '/', report['before_test']['attempts'])
print('After:', report['after_test']['successes'], '/', report['after_test']['attempts'])
print('Artifact:', '/content/biogym-run.zip')
